# Towards Generative Modeling of CO₂ Geological Storage with Latent Conditional Diffusion Models

|| **Author: Vittoria De Pellegrini** || **Year: 2025** ||
**Institution: King Abdullah University of Science and Technology (KAUST)** ||

This notebook provides a comprehensive guide for training a **VQ-VAE (Vector Quantized Variational Autoencoder) + GAN** Model for CO₂ Gas Saturation Dataset.

### ⚠️ Define Normalization Strategy

To normalize our dataset to the range $[-1, 1]$, we utilize **Local Normalization**. This approach involves normalizing each sample independently based on its own minimum and maximum values. We adopt local normalization because the range of values differs between samples. The formula for local normalization is calculated as follows: 

$$x_{\text{norm}} = 2 \times \frac{x - \min(x)}{\max(x) - \min(x)} - 1\tag{1}$$

Where:
- $\min(x)$ and $\max(x)$ are the minimum and maximum values of the each sample $x$ in the dataset $X = \{x_1, x_2, \dots, x_N\}$, where $N$ is the number of samples.
- $x_{\text{norm}}$ denotes the normalized value, which will falls in the range $[-1, 1]$.

Note: 
- For the Condition (porosity) we still apply **Global Normalization**, and we use it for plotting purposes.

### 📚⬇ Import Libraries and Functions 

In [1]:
# Import libraries 
import warnings
warnings.filterwarnings('ignore')

import os 
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import optuna 

import accelerate 
from accelerate import notebook_launcher

import torch
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

import sys
sys.path.append('/.../')

# Import functions 
from dmfco2 import(
    set_seed,     
    Discriminator,
    LPIPS,
    VQVAE,
    trainer_vqvae,
    visualize_vqvae_reconstructed_co2_samples_train
)

### ❗️ Identify Number of Devices (Multi-GPU Training with Accelerate)

In [2]:
# Print the number of GPUs available
num_devices = torch.cuda.device_count()
print("Number of devices: ", num_devices)

# Loop through all available devices and print their names
for i in range(num_devices):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")
    
# Print current device
print("Current device: ", torch.cuda.current_device())

### ❗️Set Seed for Reproducible Results

In [3]:
# Set seed
set_seed(seed=42)

### 📘 Define Inputs (Conditions) and Outputs (Targets) Variables for CO₂ Gas Saturation Training Dataset

In [4]:
# Define training dataset directory path
data_dir_sg_train = '/.../co2_dataset/training_dataset_gas_saturation'

In [5]:
# Define inputs (conditions) and outputs (targets) variables
train_sg_inputs = torch.load(f'{data_dir_sg_train}/sg_train_inputs.pt')
train_sg_outputs = torch.load(f'{data_dir_sg_train}/sg_train_outputs.pt')

print(train_sg_inputs.shape) # Shape: [num_samples, height, width, time_steps, num_features]
print(train_sg_outputs.shape)

In [ ]:
# Inputs -> Extract porosity (index 2) and last time frame 30.0 y (index -1)
train_sg_inputs  = train_sg_inputs[:, :, :, -1, 2]  

# Outputs -> Extract last time frame 30.0 y
train_sg_outputs = train_sg_outputs[:, :, :, -1] 

print("New shape:", train_sg_inputs.shape)
print("New shape:", train_sg_outputs.shape)

In [7]:
# Add number of features (1) to inputs (conditions) and outputs (targets)
train_sg_inputs = train_sg_inputs[:, :, :].unsqueeze(1) # Shape will be [4500, 1, 96, 200]
train_sg_outputs = train_sg_outputs[:, :, :].unsqueeze(1) # Shape will be [4500, 1, 96, 200]

# Print new shape
print("New shape:", train_sg_inputs.shape)   # Expected: [num_samples, num_features, height, width] -> Only Porosity as Condition 
print("New shape:", train_sg_outputs.shape)  # Expected: [num_samples, num_features, height, width] -> Only CO₂ Gas Saturation @ year 30.0

In [8]:
# Check ranges for each sample individually (CONDITION)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_train_por_individual = train_sg_inputs.amin(dim=(1, 2, 3))  # Min per sample
max_train_por_individual = train_sg_inputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_train_por_individual: {min_train_por_individual.shape}") # [4500]
print(f"Shape of max_train_por_individual: {max_train_por_individual.shape}") # [4500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_train_por_individual, max_train_por_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")

In [9]:
# Normalize inputs between -1 and 1 
min_train_por = train_sg_inputs.min()
max_train_por = train_sg_inputs.max()

# Check min and max values in train_sg_inputs before global normalization
print("Global Min for Porosity before normalization:", train_sg_inputs.min().item(), "Global Max for Porosity before normalization:", train_sg_inputs.max().item())

# Global Normalization formula
train_sg_inputs = 2 * (train_sg_inputs - min_train_por) / (max_train_por - min_train_por) - 1

# Check min and max values in train_sg_inputs after global normalization
print("Global Min for Porosity after normalization:", train_sg_inputs.min().item(), "Global Min for Porosity after normalization:", train_sg_inputs.max().item())

In [10]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_train_sg_individual = train_sg_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_train_sg_individual = train_sg_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_train_sg_individual: {min_train_sg_individual.shape}")  # [4500]
print(f"Shape of max_train_sg_individual: {max_train_sg_individual.shape}")  # [4500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_train_sg_individual, max_train_sg_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in train_sg_outputs before local normalization
print("Global Min for CO₂ Gas Saturation before normalization:", train_sg_outputs.min().item(), "Global Max for CO₂ Gas Saturation  before normalization:", train_sg_outputs.max().item())

In [11]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
co2_log_file = os.path.join(log_dir, "train_co2_min_max_before_normalization.txt")

# Save min and max values before normalization into a text file
with open(co2_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values Before Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_train_sg_individual)):
        log_file.write(f"Sample {i+1}: Min Before={min_train_sg_individual[i].item():.4f}, Max Before={max_train_sg_individual[i].item():.4f}\n")

In [12]:
# Normalize outputs between -1 and 1 

# Initialize arrays to store min and max values for each sample
min_train_sg = []
max_train_sg = []

# Normalize outputs and store min/max values
for i in range(len(train_sg_outputs)):
    min_val = train_sg_outputs[i].min()
    max_val = train_sg_outputs[i].max()
    print(f"Sample {i + 1}:")
    print(f"Original Min: {min_val}, Original Max: {max_val}")
    min_train_sg.append(min_val)
    max_train_sg.append(max_val)

    # Local Normalization formula
    train_sg_outputs[i] = 2 * (train_sg_outputs[i] - min_val) / (max_val - min_val) - 1
    print(f"Normalized Output: {train_sg_outputs[i]}")

In [13]:
#Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_train_sg_individual_after_norm = train_sg_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_train_sg_individual_after_norm = train_sg_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_train_sg_individual_after_norm: {min_train_sg_individual_after_norm.shape}")  # [4500]
print(f"Shape of max_train_sg_individual_after_norm: {max_train_sg_individual_after_norm.shape}")  # [4500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_train_sg_individual_after_norm, max_train_sg_individual_after_norm)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in train_sg_outputs after local normalization
print("Global Min for CO₂ Gas Saturation after normalization:", train_sg_outputs.min().item(), "Global Min for CO₂ Gas Saturation after normalization:", train_sg_outputs.max().item())

In [14]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
co2_log_file = os.path.join(log_dir, "train_co2_min_max_after_normalization.txt")

# Save min and max values before normalization into a text file
with open(co2_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values After Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_train_sg_individual_after_norm)):
        log_file.write(f"Sample {i+1}: Min After={min_train_sg_individual_after_norm[i].item():.4f}, Max After={max_train_sg_individual_after_norm[i].item():.4f}\n")

In [15]:
# Define variables for conditional diffusion model
train_sg_label = train_sg_inputs
train_sg_target = train_sg_outputs

### 🗂️ Define Training Dataset and Dataloader

In [16]:
# Create dataloader
training_dataset = TensorDataset(train_sg_target, train_sg_label)

batch_size = 32

training_dataloader = DataLoader(training_dataset, 
                                 batch_size=batch_size, 
                                 shuffle=True,
                                 drop_last=True, # To remove the batch with less samples
                                 pin_memory=True, # Optimizes data transfer to GPU
                                 #num_workers=4 # Parallel data loading
                                 ) 

# Check the data sizes
for feat, lab in training_dataloader:
    print(f'Features shape: {feat.shape} \t Labels shape: {lab.shape}')
    break

### 🚀 Preparing for VQ-VAE Training with Optuna and Accelerator: CO₂ Gas Saturation

#### 01. Preliminary Training with Optuna and Accelerator
The goal is to identify the optimal combination of hyperparameters using Optuna for efficient hyperparameter tuning, and Accelerator to speed up training time. For quick evaluation we use few epochs. 


In [17]:
# Define the objective function for Optuna hyperparameter optimization
def objective(trial):
    
    # Define hyperparameter search space
    z_channels = trial.suggest_categorical("z_channels", [
    2, 3, 4, 5, 6, 7, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 
    36, 40, 44, 48, 52, 56, 60, 64, 72, 80, 88, 96, 104, 112, 120, 128, 
    136, 144, 152, 160, 176, 192, 208, 224, 240, 256, 288, 320, 352, 384, 
    416, 448, 480, 512, 576, 640, 704, 768
])
    codebook_size = trial.suggest_categorical("codebook_size", [
    64, 80, 96, 112, 128, 144, 160, 192, 224, 256, 288, 320, 352, 384, 
    416, 448, 480, 512, 576, 640, 704, 768, 832, 896, 960, 1024, 1152, 
    1280, 1408, 1536, 1664, 1792, 1920, 2048, 2304, 2560, 2816, 3072, 
    3328, 3584, 3840, 4096, 4608, 5120, 5632, 6144, 6656, 7168, 7680, 
    8192, 9216, 10240, 11264, 12288, 13312, 14336, 15360, 16384, 18432, 
    20480, 22528, 24576, 28672, 32768, 36864, 40960, 49152, 57344, 65536
])
    num_epochs = trial.suggest_categorical("num_epochs", [10])  # Fewer epochs for quick evaluation
    disc_step_start = trial.suggest_categorical("disc_step_start", [50, 100, 200, 500, 750])
    
    # Loss Weights
    recon_loss_weight = trial.suggest_categorical("recon_loss_weight", [1]) # We keep it equal to 1.0
    codebook_loss_weight = trial.suggest_float("codebook_loss_weight", 0.1, 0.5, step=0.05)
    commitment_loss_weight = trial.suggest_float("commitment_loss_weight", 0.05, 0.3, step=0.05)
    disc_fake_loss_weight = trial.suggest_float("disc_fake_loss_weight", 0.1, 0.7, step=0.1)
    disc_loss_weight = trial.suggest_float("disc_loss_weight", 0.5, 0.7, step=0.1)
    lpips_loss_weight = trial.suggest_categorical("lpips_loss_weight", [1]) # We keep it equal to 1.0
    
    # Optimizer learning rates
    lr_g = trial.suggest_float("lr_g", 1e-6, 2e-5, log=True) # search on a logarithmic scale
    lr_d = trial.suggest_float("lr_d", 1e-5, 1e-4, log=True) # search on a logarithmic scale
    
    # Print trial details 
    print(f"Trial {trial.number}: Parameters - {trial.params}")

    # Initialize model and components
    vqvae_model = VQVAE(z_channels=z_channels, codebook_size=codebook_size) # Define VQ-VAE Model
    discriminator = Discriminator(im_channels=1) # Define Discriminator 
    lpips_model = LPIPS().eval() # Define LPIPS Model
    recon_criterion = torch.nn.MSELoss()
    disc_criterion = torch.nn.BCEWithLogitsLoss()
    optimizer_g = torch.optim.Adam(vqvae_model.parameters(), lr=lr_g, betas=(0.5, 0.999))
    optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=lr_d, betas=(0.5, 0.999))

    # Temporary directory for trial models
    trial_save_dir = f"/.../trained_models/gas_saturation_models/vqvae/trial_model_{trial.number}"
    os.makedirs(trial_save_dir, exist_ok=True)
    summary_file_path = f"/.../trained_models/gas_saturation_models/vqvae/trial_model_{trial.number}/vqvae_training_summary.txt"

    # Conduct partial training to evaluate hyperparameters
    g_loss_list, _ = trainer_vqvae(
        model=vqvae_model,
        lpips_model=lpips_model,
        discriminator=discriminator,
        recon_criterion=recon_criterion,
        disc_criterion=disc_criterion,
        optimizer_g=optimizer_g,
        optimizer_d=optimizer_d,
        training_dataloader=training_dataloader,
        num_epochs=num_epochs,  # Limited training epochs
        disc_step_start=disc_step_start,
        save_dir=trial_save_dir,
        save_dir_results=None,
        summary_file_path=summary_file_path,
        trial_params=trial.params,
        save_loss_curves=False,
        loss_weights={
            "recon_loss": recon_loss_weight,
            "codebook_loss": codebook_loss_weight,
            "commitment_loss": commitment_loss_weight,
            "disc_fake_loss": disc_fake_loss_weight,
            "disc_loss": disc_loss_weight,
            "lpips_loss": lpips_loss_weight,
        },
        early_stopping_patience=10
        )

    # Use the final generator loss as the optimization target
    return g_loss_list[-1]  # Last generator loss of partial training

In [ ]:
# Save Best Hyperparameters into JSON file 

# Define the storage for Optuna to enable dashboard tracking
storage_url = "sqlite:////.../trained_models/gas_saturation_models/vqvae/optuna_vqvae.db"

# File path to save the best trial parameters
best_trial_file = "/.../trained_models/gas_saturation_models/vqvae/best_model/best_trial_params_partial_training.json"
best_trial_dir = "/.../trained_models/gas_saturation_models/vqvae/best_model"
final_model_path = os.path.join(best_trial_dir, "vqvae_model_partial_training.pth")

# Ensure the directory exists
os.makedirs(best_trial_dir, exist_ok=True)

# Define total trials (make sure this is your actual target)
total_trials = 500

# Define the optimization process using Optuna
def optimization_process():
    global study
    study = optuna.create_study(direction="minimize", study_name="vqvae_hyperparam_tuning", storage=storage_url, load_if_exists=True, sampler=optuna.samplers.TPESampler()) # Minimize generator loss
    
    # Check completed trials
    completed_trials = len(study.trials)
    remaining_trials = max(0, total_trials - completed_trials)  # Avoid negative values
    
    print(f"Total Trials Target: {total_trials}")
    print(f"Completed Trials: {completed_trials}")
    print(f"Remaining Trials: {remaining_trials}")

    if remaining_trials > 0:
        # Define callback to save the best trial parameters after each trial
        def save_best_trial(study, trial):
            best_trial = study.best_trial
            print(f"Best trial number: {best_trial.number}")
            print(f"Best trial hyperparameters: {best_trial.params}")
            
            # Save the best trial parameters to a file
            with open(best_trial_file, "w") as f:
                json.dump(best_trial.params, f)
            
            print(f"Best trial parameters (Trial {best_trial.number}) saved to {best_trial_file}")
            
            # Now save the best model checkpoint
            best_z_channels = best_trial.params["z_channels"]
            best_codebook_size = best_trial.params["codebook_size"]
            
            # Recreate the best model with the optimized parameters
            vqvae_model = VQVAE(z_channels=best_z_channels, codebook_size=best_codebook_size)

            # Assuming vqvae_model is already trained and defined globally
            torch.save({
                "z_channels": best_z_channels,
                "codebook_size": best_codebook_size,
                "state_dict": vqvae_model.state_dict()
            }, final_model_path)

            print(f"Best model checkpoint saved to {final_model_path}")
    
        # Run the optimization and attach the callback
        study.optimize(objective, n_trials=remaining_trials, callbacks=[save_best_trial]) 
    else:
        print("All trials are already completed. No additional trials will be run.")
        
# Run the optimization process using notebook_launcher for multi-GPU setups
notebook_launcher(optimization_process, args=(), num_processes=1)

### How to Activate Optuna Dashboard ###
#Copy paste this line into your terminal, after going inside the correct directory: optuna-dashboard sqlite:///optuna_vqvae.db
#Or directly copy paste into your terminal: optuna-dashboard sqlite:////.../trained_models/gas_saturation_models/vqvae/optuna_vqvae.db
#Go to VSC Command Palette and then “Simple Browser: Show” and copy paste the URL link: http://127.0.0.1:8080

#### 02. Full Training with Optuna and Accelerator

After identifying the best hyperparameters, we train the VQ-VAE+GAN model on an increased number of epochs.

##### 02a. Set-Up Full Training

In [ ]:
# Load best trial parameters
best_trial_file = "/.../trained_models/gas_saturation_models/vqvae/best_model/best_trial_params_partial_training.json"
with open(best_trial_file, "r") as f:
    best_trial_params = json.load(f)

# Initialize model with the best hyperparameters
best_z_channels = best_trial_params["z_channels"]
best_codebook_size = best_trial_params["codebook_size"]
best_disc_step_start = best_trial_params["disc_step_start"]

# Loss Weights
best_recon_loss_weight = best_trial_params["recon_loss_weight"]
best_codebook_loss_weight = best_trial_params["codebook_loss_weight"] 
best_commitment_loss_weight = best_trial_params["commitment_loss_weight"]    
best_disc_fake_loss_weight = best_trial_params["disc_fake_loss_weight"]   
best_disc_loss_weight = best_trial_params["disc_loss_weight"]    
best_lpips_loss_weight = best_trial_params["lpips_loss_weight"]  

# Optimizer learning rates
best_lr_g = best_trial_params["lr_g"] 
best_lr_d = best_trial_params["lr_d"] 

print("Best trial parameters loaded successfully.")

# Print best hyperparameters
print(f"Best z_channels: {best_z_channels}")
print(f"Best codebook_size: {best_codebook_size}")
print(f"Best disc_step_start: {best_disc_step_start}")

print(f"Best recon_loss_weight: {best_recon_loss_weight}")
print(f"Best codebook_loss_weight: {best_codebook_loss_weight}")
print(f"Best commitment_loss_weight: {best_commitment_loss_weight}")
print(f"Best disc_fake_loss_weight: {best_disc_fake_loss_weight}")
print(f"Best disc_loss_weight: {best_disc_loss_weight}")
print(f"Best lpips_loss_weight: {best_lpips_loss_weight}")

print(f"Best lr_g: {best_lr_g}")
print(f"Best lr_d: {best_lr_d}")

# Define number of epochs
num_epochs=300

# Initialize model and components
vqvae_model = VQVAE(z_channels=best_z_channels, codebook_size=best_codebook_size)
discriminator = Discriminator(im_channels=1)
lpips_model = LPIPS().eval()
recon_criterion = torch.nn.MSELoss()
disc_criterion = torch.nn.BCEWithLogitsLoss()
optimizer_g = torch.optim.Adam(vqvae_model.parameters(), lr=best_lr_g, betas=(0.5, 0.999))
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=best_lr_d, betas=(0.5, 0.999))

# Directory for final model and results
final_save_dir = "/.../trained_models/gas_saturation_models/vqvae/best_model"
os.makedirs(final_save_dir, exist_ok=True)
final_summary_path = "/.../trained_models/gas_saturation_models/vqvae/best_model/vqvae_training_summary_full.txt"

# Define save directory for loss curves
save_dir_results = '/.../results/vqvae_gas_saturation_results'
os.makedirs(save_dir_results, exist_ok=True)

# Full training with best hyperparameters
g_loss_list, disc_loss_list = trainer_vqvae(
    model=vqvae_model,
    lpips_model=lpips_model,
    discriminator=discriminator,
    recon_criterion=recon_criterion,
    disc_criterion=disc_criterion,
    optimizer_g=optimizer_g,
    optimizer_d=optimizer_d,
    training_dataloader=training_dataloader,
    num_epochs=num_epochs,  # Full training with more epochs
    disc_step_start=best_disc_step_start,
    save_dir=final_save_dir,
    save_dir_results=save_dir_results,
    summary_file_path=final_summary_path,
    trial_params=best_trial_params,  # Save best trial parameters in full training summary
    save_loss_curves=True,
    loss_weights={
            "recon_loss": best_recon_loss_weight,
            "codebook_loss": best_codebook_loss_weight,
            "commitment_loss": best_commitment_loss_weight,
            "disc_fake_loss": best_disc_fake_loss_weight,
            "disc_loss": best_disc_loss_weight,
            "lpips_loss": best_lpips_loss_weight,
        },
    early_stopping_patience=10
)

# Save the final trained model along with initialization parameters
torch.save({
    "z_channels": best_z_channels,
    "codebook_size": best_codebook_size,
    "state_dict": vqvae_model.state_dict()
}, os.path.join(final_save_dir, "vqvae_model.pth"))

# Save the final trained discriminator
torch.save(discriminator.state_dict(), os.path.join(final_save_dir, "vqvae_discriminator.pth"))
print("Final trained model and discriminator saved.")

### How to Activate TensorBoard ###
#Once traning is started copy paste this line into your terminal: tensorboard --logdir=/.../results/vqvae_gas_saturation_results/tensorboard_logs --port=6002
#Go to VSC Command Palette and then “Simple Browser: Show” and copy paste the URL link: http://127.0.0.1:6002

##### 02b. Save Reconstructed Samples

In [20]:
# Load saved model
vqvae_checkpoint = '/.../trained_models/gas_saturation_models/vqvae/best_model/vqvae_model.pth' 
vqvae_checkpoint = torch.load(vqvae_checkpoint)

# Extract parameters and weights
z_channels = vqvae_checkpoint["z_channels"]
codebook_size = vqvae_checkpoint["codebook_size"]
state_dict = vqvae_checkpoint["state_dict"]

# Initialize the model with the saved parameters
vqvae_model = VQVAE(z_channels=z_channels, codebook_size=codebook_size)

# Load the model weights
vqvae_model.load_state_dict(state_dict)

print("Model loaded successfully with z_channels={} and codebook_size={}".format(z_channels, codebook_size))

In [ ]:
# Define save directory for loss curves
save_dir = "/.../results/vqvae_gas_saturation_results/reconstructed_samples_training_dataset"

# Specify the first 4500 sample indices
sample_indices = list(range(4500))

# Call the function for CO₂ gas saturation
visualize_vqvae_reconstructed_co2_samples_train(
    model=vqvae_model,
    dataset=training_dataset,
    sample_indices=sample_indices,
    min_train_sg=min_train_sg,
    max_train_sg=max_train_sg,
    min_train_por=min_train_por,
    max_train_por=max_train_por,
    save_dir=save_dir
)

## References 

**[1] Wen et al., 2002** ([Paper](https://www.sciencedirect.com/science/article/pii/S0309170822000562)): CO₂ Open-Source Dataset ([Google Drive Link](https://drive.google.com/drive/folders/1fZQfMn_vsjKUXAfRV0q_gswtl8JEkVGo)).

[2] How to train a VQ-VAE Model ([GitHub Link](https://github.com/explainingai-code/StableDiffusion-PyTorch/tree/main)).